# Moderación semiautomática de videos peruanos de YouTube mediante modelos clásicos y neuronales de procesamiento del lenguaje natural

**Trabajo final del curso de Procesamiento de Lenguaje Natural (PLN) de la Maestría en Inteligencia Artificial de la Universidad Nacional de Ingeniería (UNI) — Semestre 2026-1**

**Grupo 4:** Luis Enrique Koc Góngora, Alex Felipe Mancilla Antay, Herbert Antonio Meléndez García y Dennis Jack Paitán Cano

---

## 02.00 · Publicación del bundle versionado desde Colab

Descarga de GitHub el bundle sincronizado —o permite cargar el bundle local más reciente—, verifica su identidad y lo publica en Google Drive mediante la autorización integrada de Colab.

La identidad estable combina los SHA-256 del código y de las entradas comprimidas. El cuaderno verifica el `bundle_id` y cada artefacto antes de escribir en Drive; publica los artefactos y deja el manifiesto y `latest.json` para el final, de modo que una interrupción no se presente como una versión completa [1]. El almacenamiento de la VM de Colab es efímero y el montaje de Drive solicita autorización integrada durante la sesión [2]. La revisión Git, la carpeta de Drive y el momento de publicación son decisiones locales.

**Contrato de etiquetas v2.1:** cinco salidas entrenadas: `SEGURO`, `RACISMO_DISCRIMINACION`, `ATAQUE_POR_GENERO_IDENTIDAD`, `ACOSO_AMENAZA` y `CONTENIDO_SEXUAL`. `SEGURO` es excluyente; las cuatro categorías de daño son multietiqueta y pueden coexistir. Los casos indeterminados se difieren y no entran al entrenamiento. Esta combinación, sus umbrales y sus reglas de exclusividad son decisiones operativas locales.

## Reproducibilidad

El cuaderno obtiene un bundle ya construido desde GitHub o desde el selector de archivos del navegador, verifica su identidad y todos sus SHA-256, y solo entonces publica una versión inmutable en Drive. No requiere Google Cloud Console ni Drive Desktop.

## Backend Google Colab

Abra este archivo en **Google Colab** y ejecute sus celdas en orden. El modo `github` descarga el bundle sincronizado de la revisión configurada; `local_upload` permite escoger todos los archivos del bundle local más reciente. La autorización integrada de `drive.mount()` publica en `Mi unidad/ModeracionPeru_Colab` [2].

In [ ]:
# Este cuaderno se ejecuta en Google Colab y no requiere Google Cloud Console.
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import importlib.util
import json
import os
import shutil
import urllib.parse
import urllib.request
import uuid

from IPython.display import Markdown, display
from tqdm.auto import tqdm

if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError(
        "02_00 debe abrirse y ejecutarse en Google Colab. La carga local usa el selector del navegador."
    )

COLAB_EXPECTED_CORE_SHA256 = "a5e4301aabbb195ff0d1685efc15c5fe51244e6115234c81273a28fe07bad3f3"
COLAB_NOTEBOOK_BUILD_BUNDLE_ID = "9ef0424dae1e9289e33609b5cf830d0aad1e21adc3b1f731847e7988a7c6847a"

def _json_text(value):
    return json.dumps(value, ensure_ascii=False, indent=2, default=str)

def show_result(title, value, tone="success"):
    display(Markdown(f"### {title}\n\n```json\n{_json_text(value)}\n```"))

def show_summary(title, value, tone="neutral"):
    show_result(title, value, tone=tone)

def show_callout(title, message, tone="neutral"):
    display(Markdown(f"> **{title}.** {message}"))

def _sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        while block := handle.read(1024 * 1024):
            digest.update(block)
    return digest.hexdigest()

def _read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))

def _bundle_id_for_manifest(manifest):
    core = manifest["core"]
    inputs = manifest["inputs"]
    identity = {
        "schema_version": manifest["schema_version"],
        "taxonomy_contract": manifest["taxonomy_contract"],
        "taxonomy_version": manifest["taxonomy_version"],
        "core": {"name": core["name"], "sha256": core["sha256"]},
        "inputs": {
            key: {
                "archive": value["archive"],
                "archive_sha256": value["archive_sha256"],
                "source_sha256": value["source_sha256"],
            }
            for key, value in sorted(inputs.items())
        },
    }
    payload = json.dumps(identity, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()

def _bundle_specs(manifest):
    specs = [(manifest["core"]["name"], manifest["core"]["sha256"])]
    specs.extend(
        (entry["archive"], entry["archive_sha256"])
        for entry in manifest.get("inputs", {}).values()
    )
    for name, expected_sha256 in specs:
        if Path(name).name != name or len(str(expected_sha256)) != 64:
            raise ValueError(f"Entrada insegura o incompleta en bundle_manifest.json: {name!r}")
    return [(str(name), str(expected_sha256)) for name, expected_sha256 in specs]

def _verify_bundle(directory, expected_bundle_id=None):
    bundle_dir = Path(directory)
    manifest_path = bundle_dir / "bundle_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Falta {manifest_path}")
    manifest = _read_json(manifest_path)
    computed = _bundle_id_for_manifest(manifest)
    if manifest.get("bundle_id") != computed:
        raise ValueError("bundle_manifest.json no contiene un bundle_id válido")
    if expected_bundle_id is not None and computed != expected_bundle_id:
        raise ValueError(f"Bundle inesperado: esperado={expected_bundle_id}, obtenido={computed}")
    if manifest["core"]["sha256"] != COLAB_EXPECTED_CORE_SHA256:
        raise RuntimeError(
            "El core del bundle no coincide con este 02_00. Descargue el cuaderno y el bundle de la misma revisión."
        )
    for name, expected_sha256 in _bundle_specs(manifest):
        artifact = bundle_dir / name
        if not artifact.is_file():
            raise FileNotFoundError(f"Falta el artefacto declarado {artifact}")
        actual = _sha256(artifact)
        if actual != expected_sha256:
            raise ValueError(f"SHA-256 inválido para {name}: esperado={expected_sha256}, obtenido={actual}")
    return manifest

show_summary(
    "Entorno publicador",
    {
        "backend": "Google Colab",
        "Google Cloud Console": "no requerido",
        "Drive Desktop": "no requerido",
        "core esperado": COLAB_EXPECTED_CORE_SHA256,
        "bundle al generar el cuaderno": COLAB_NOTEBOOK_BUILD_BUNDLE_ID,
    },
    tone="success",
)


## Configuración de la fuente y el destino

In [ ]:
RUN_PUBLISH_BUNDLE=False  # Cambie a True después de revisar esta celda.
BUNDLE_SOURCE='github'     # 'github' o 'local_upload'.
# github: descarga la versión ya sincronizada; local_upload: selector del navegador.
GITHUB_REPOSITORY='lkoc/Trabajo_PLN-MIA-Grupo4'
GITHUB_REF='main'          # Rama, etiqueta o commit reproducible.
GITHUB_BUNDLE_PATH='resultados/colab_bundle'
COLAB_DRIVE_FOLDER='ModeracionPeru_Colab'
STAGING=Path('/content/moderacion_peru_bundle_source')
REQUIRED_BUNDLE_FILES=('project_core.zip','chunks_v2.jsonl.gz','chunks_deepseek_historicos.jsonl.gz','deepseek_flash_historico.jsonl.gz','deepseek_pro_historico_principal.jsonl.gz','deepseek_pro_historico_umbral.jsonl.gz','deepseek_pro_historico_sospechosos.jsonl.gz','dataset_5_salidas.jsonl.gz','bundle_manifest.json')

if BUNDLE_SOURCE not in {'github','local_upload'}:
    raise ValueError("BUNDLE_SOURCE debe ser 'github' o 'local_upload'.")
show_summary('Publicación preparada',{'ejecutar':RUN_PUBLISH_BUNDLE,'fuente':BUNDLE_SOURCE,'repositorio':GITHUB_REPOSITORY if BUNDLE_SOURCE=='github' else None,'revisión':GITHUB_REF if BUNDLE_SOURCE=='github' else None,'archivos_locales_requeridos':REQUIRED_BUNDLE_FILES if BUNDLE_SOURCE=='local_upload' else None,'destino':f'Mi unidad/{COLAB_DRIVE_FOLDER}/bundle_releases'},tone='neutral')
show_callout('Dos momentos de ejecución','Ejecute 02_00 antes de 02_01 para publicar chunks y vuelva a ejecutarlo después de 02_05 para publicar el snapshot que consumirá la etapa 03.',tone='info')

## Adquisición y verificación integral

In [ ]:
def _download_file(url, destination):
    request=urllib.request.Request(url,headers={'User-Agent':'ModeracionPeru-Colab-Bundle/1.0'})
    with urllib.request.urlopen(request,timeout=120) as response, destination.open('wb') as target:
        total=int(response.headers.get('Content-Length') or 0)
        with tqdm(total=total or None,desc=f'Descargando {destination.name}',unit='B',unit_scale=True) as bar:
            while block:=response.read(1024*1024):
                target.write(block); bar.update(len(block))

def _prepare_staging():
    if STAGING != Path('/content/moderacion_peru_bundle_source'):
        raise RuntimeError('STAGING debe permanecer dentro del espacio efímero controlado de Colab.')
    if STAGING.exists():
        shutil.rmtree(STAGING)
    STAGING.mkdir(parents=True)

if RUN_PUBLISH_BUNDLE:
    _prepare_staging()
    if BUNDLE_SOURCE=='github':
        encoded_ref=urllib.parse.quote(GITHUB_REF,safe='')
        base=f'https://raw.githubusercontent.com/{GITHUB_REPOSITORY}/{encoded_ref}/{GITHUB_BUNDLE_PATH}'
        for name in ('bundle_manifest.json',*REQUIRED_BUNDLE_FILES[:-1]):
            _download_file(f'{base}/{name}',STAGING/name)
    else:
        from google.colab import files
        show_callout('Seleccione nueve archivos','Abra resultados/colab_bundle en su PC y seleccione simultáneamente los nueve archivos indicados. El navegador es el puente; Colab no puede leer D: directamente.',tone='info')
        uploaded=files.upload()
        missing=sorted(set(REQUIRED_BUNDLE_FILES)-set(uploaded))
        if missing:
            raise FileNotFoundError(f'Faltaron archivos en la carga local: {missing}')
        unexpected=sorted(set(uploaded)-set(REQUIRED_BUNDLE_FILES))
        if unexpected:
            show_callout('Archivos adicionales ignorados',str(unexpected),tone='warning')
        for name in REQUIRED_BUNDLE_FILES:
            (STAGING/name).write_bytes(uploaded[name])
    bundle_manifest=_verify_bundle(STAGING)
    show_result('Bundle adquirido y verificado',{'bundle_id':bundle_manifest['bundle_id'],'core_sha256':bundle_manifest['core']['sha256'],'fuente':BUNDLE_SOURCE,'archivos':{name:{'bytes':(STAGING/name).stat().st_size,'sha256':_sha256(STAGING/name)} for name in REQUIRED_BUNDLE_FILES}},tone='success')
else:
    bundle_manifest=None
    show_callout('Adquisición desactivada','Revise la configuración y cambie RUN_PUBLISH_BUNDLE=True. Aún no se descargó ni cargó nada.',tone='neutral')

## Publicación inmutable y actualización de latest.json

In [ ]:
def _copy_with_progress(source,destination):
    with source.open('rb') as raw, destination.open('wb') as target, tqdm(total=source.stat().st_size,desc=f'Publicando {source.name}',unit='B',unit_scale=True) as bar:
        while block:=raw.read(1024*1024):
            target.write(block); bar.update(len(block))

if RUN_PUBLISH_BUNDLE:
    from google.colab import drive
    drive.mount('/content/drive',force_remount=False)
    DRIVE_ROOT=Path('/content/drive/MyDrive')/COLAB_DRIVE_FOLDER
    RELEASES_DIR=DRIVE_ROOT/'bundle_releases'
    RELEASES_DIR.mkdir(parents=True,exist_ok=True)
    bundle_id=str(bundle_manifest['bundle_id'])
    release_dir=RELEASES_DIR/bundle_id
    specs=_bundle_specs(bundle_manifest)
    if release_dir.exists():
        _verify_bundle(release_dir,expected_bundle_id=bundle_id)
        release_status='already_present_and_verified'
    else:
        partial=RELEASES_DIR/f'.{bundle_id}.partial-{uuid.uuid4().hex}'
        partial.mkdir()
        try:
            for name,_ in specs:
                _copy_with_progress(STAGING/name,partial/name)
            _copy_with_progress(STAGING/'bundle_manifest.json',partial/'bundle_manifest.json')
            _verify_bundle(partial,expected_bundle_id=bundle_id)
            os.replace(partial,release_dir)
        finally:
            if partial.exists():
                shutil.rmtree(partial)
        _verify_bundle(release_dir,expected_bundle_id=bundle_id)
        release_status='published_and_verified'
    pointer={'schema_version':'1.0.0','bundle_id':bundle_id,'core_sha256':bundle_manifest['core']['sha256'],'manifest_sha256':_sha256(release_dir/'bundle_manifest.json'),'published_at':datetime.now(timezone.utc).isoformat()}
    latest_path=RELEASES_DIR/'latest.json'
    partial_latest=RELEASES_DIR/f'.latest-{uuid.uuid4().hex}.json'
    partial_latest.write_text(json.dumps(pointer,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
    os.replace(partial_latest,latest_path)
    persisted_pointer=_read_json(latest_path)
    if persisted_pointer!=pointer:
        raise RuntimeError('latest.json no conservó exactamente el puntero publicado')
    bundle_result={'status':'published_to_drive','release_status':release_status,'bundle_id':bundle_id,'release_dir':str(release_dir),'latest_pointer':str(latest_path),'manifest_sha256':pointer['manifest_sha256']}
    show_result('Versión de Colab publicada en Drive',bundle_result,tone='success')
    show_callout('Siguiente paso','02_01 y 03_02–03_06 resolverán latest.json, verificarán esta versión y la activarán antes de importar el proyecto.',tone='success')
else:
    show_callout('Publicación desactivada','No se montó Drive ni se escribió ningún archivo.',tone='neutral')

## Referencias

[1] National Institute of Standards and Technology, "Secure Hash Standard (SHS)," FIPS PUB 180-4, Aug. 2015, doi: 10.6028/NIST.FIPS.180-4.

[2] Google Colab, "Frequently Asked Questions," Google Research, 2026. [Online]. Available: https://research.google.com/colaboratory/faq.html. Accessed: Aug. 7, 2026.